# Model 3 — HistGradientBoosting Regressor

This notebook trains **Model 3: HistGradientBoostingRegressor** using the same Manhattan + Queens train/test split and modelling features as Models 1 and 2.

### Why this model?
- It is a strong nonlinear tree-based model for large tabular datasets.
- The model supports **Poisson loss**, which is appropriate for a non-negative count target such as `trip_count`.
- Categorical variables are handled as categorical features after ordinal encoding rather than one-hot encoding.

### Important
This is the **first untuned Model 3 run**. We will compare all three models before deciding whether tuning is necessary. Scikit-learn documents Poisson loss as requiring `y >= 0`, and HistGradientBoosting supports categorical features; the model is designed to scale well to datasets with many samples. citeturn998900search0turn998900search1


## 1. Imports and configuration


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Locate the repository and load the prepared data


In [14]:
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "models" / "modeling" / "train.csv").exists():
            return path
    raise FileNotFoundError(
        "Could not find models/modeling/train.csv. "
        "Run modeling_preparation_baseline.ipynb first."
    )

REPO_ROOT = find_repo_root()
MODELING_DIR = REPO_ROOT / "models" / "modeling"

TRAIN_PATH = MODELING_DIR / "train.csv"
TEST_PATH = MODELING_DIR / "test.csv"
BASELINE_METRICS_PATH = MODELING_DIR / "baseline_metrics.csv"
MODEL_1_METRICS_PATH = MODELING_DIR / "model_1_metrics.csv"
MODEL_2_METRICS_PATH = MODELING_DIR / "model_2_metrics.csv"

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["pickup_hour"])
test_df = pd.read_csv(TEST_PATH, parse_dates=["pickup_hour"])

print("Repository:", REPO_ROOT)
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train period:", train_df["pickup_hour"].min(), "->", train_df["pickup_hour"].max())
print("Test period :", test_df["pickup_hour"].min(), "->", test_df["pickup_hour"].max())


Repository: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting
Train shape: (460920, 17)
Test shape : (115368, 17)
Train period: 2025-09-08 00:00:00 -> 2026-01-25 03:00:00
Test period : 2026-01-25 04:00:00 -> 2026-02-28 23:00:00


## 3. Define target and features

The feature information is kept consistent with Models 1 and 2. For HistGradientBoosting, the categorical columns are ordinal-encoded and then explicitly marked as categorical. Ordinal encoding is used here because the estimator accepts categorical-feature indices, while the underlying model treats those features as unordered categories rather than as meaningful numeric distances. citeturn998900search1


In [15]:
TARGET = "trip_count"

CATEGORICAL_FEATURES = [
    "PULocationID",
    "hour",
    "day_of_week",
    "month",
]

NUMERICAL_FEATURES = [
    "apparent_temperature",
    "rain",
    "snowfall",
    "precipitation_probability",
    "is_holiday",
    "trip_count_lag_1h",
    "trip_count_lag_24h",
    "trip_count_lag_168h",
    "trip_count_roll_3h",
    "trip_count_roll_24h",
]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

missing_required = [
    c for c in [TARGET] + FEATURES
    if c not in train_df.columns or c not in test_df.columns
]

if missing_required:
    raise ValueError(f"Missing required modelling columns: {missing_required}")

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("Categorical features:", CATEGORICAL_FEATURES)
print("Numerical features:", NUMERICAL_FEATURES)


Categorical features: ['PULocationID', 'hour', 'day_of_week', 'month']
Numerical features: ['apparent_temperature', 'rain', 'snowfall', 'precipitation_probability', 'is_holiday', 'trip_count_lag_1h', 'trip_count_lag_24h', 'trip_count_lag_168h', 'trip_count_roll_3h', 'trip_count_roll_24h']


## 4. Validate the target and chronology


In [16]:
assert train_df["pickup_hour"].max() < test_df["pickup_hour"].min(),     "Train/test chronology is not valid."

assert (y_train >= 0).all(), "Training target contains negative values."
assert (y_test >= 0).all(), "Test target contains negative values."

print("Chronological split check: passed")
print("Train target range:", y_train.min(), "to", y_train.max())
print("Test target range :", y_test.min(), "to", y_test.max())


Chronological split check: passed
Train target range: 0 to 1122
Test target range : 0 to 905


## 5. Build the preprocessing + HistGradientBoosting pipeline

Initial settings are intentionally moderate rather than tuned. `early_stopping=False` is used so that the model does not create an internal random validation split; the project already has a fixed chronological train/test design.


In [17]:
try:
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan,
        encoded_missing_value=np.nan,
    )
except TypeError:
    # Compatibility fallback for older scikit-learn versions.
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan,
    )

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", ordinal_encoder),
])

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ("numerical", numerical_pipeline, NUMERICAL_FEATURES),
], remainder="drop")

# Categorical columns are placed first by the ColumnTransformer.
categorical_indices = list(range(len(CATEGORICAL_FEATURES)))

hgb = HistGradientBoostingRegressor(
    loss="poisson",
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    max_bins=255,
    categorical_features=categorical_indices,
    early_stopping=False,
    random_state=RANDOM_STATE,
)

model_3 = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", hgb),
])

print(model_3)
print("Categorical feature indices after preprocessing:", categorical_indices)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=nan))]),
                                                  ['PULocationID', 'hour',
                                                   'day_of_week', 'month']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))...
                 

## 6. Train Model 3


In [18]:
print("Training HistGradientBoosting Regressor...")
model_3.fit(X_train, y_train)
print("Training completed.")
print("Boosting iterations used:", model_3.named_steps["regressor"].n_iter_)


Training HistGradientBoosting Regressor...
Training completed.
Boosting iterations used: 200


## 7. Generate predictions


In [19]:
train_pred = np.clip(model_3.predict(X_train), 0, None)
test_pred = np.clip(model_3.predict(X_test), 0, None)

print("Train prediction range:", train_pred.min(), "to", train_pred.max())
print("Test prediction range :", test_pred.min(), "to", test_pred.max())


Train prediction range: 0.004757670203660995 to 934.728198303997
Test prediction range : 0.004924904805767611 to 861.8543028536905


## 8. Evaluate Model 3


In [20]:
def calculate_metrics(y_true, y_pred, dataset_name):
    return {
        "dataset": dataset_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "rows": len(y_true),
    }

model_3_metrics = pd.DataFrame([
    calculate_metrics(y_train, train_pred, "train"),
    calculate_metrics(y_test, test_pred, "test"),
])

display(model_3_metrics)


,dataset,MAE,RMSE,R2,rows
0,train,5.538155,12.870642,0.972499,460920
1,test,5.784115,13.163182,0.963343,115368


## 9. Compare Model 3 with the baseline, Model 1 and Model 2


In [21]:
comparison_rows = []

if BASELINE_METRICS_PATH.exists():
    baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH).iloc[0]
    comparison_rows.append({
        "model": "Naive 168-hour historical baseline",
        "MAE": baseline_metrics["MAE"],
        "RMSE": baseline_metrics["RMSE"],
        "R2": baseline_metrics["R2"],
    })

if MODEL_1_METRICS_PATH.exists():
    model_1_metrics = pd.read_csv(MODEL_1_METRICS_PATH)
    model_1_test = model_1_metrics[model_1_metrics["dataset"] == "test"].iloc[0]
    comparison_rows.append({
        "model": "Model 1 - Linear Regression",
        "MAE": model_1_test["MAE"],
        "RMSE": model_1_test["RMSE"],
        "R2": model_1_test["R2"],
    })

if MODEL_2_METRICS_PATH.exists():
    model_2_metrics = pd.read_csv(MODEL_2_METRICS_PATH)
    model_2_test = model_2_metrics[model_2_metrics["dataset"] == "test"].iloc[0]
    comparison_rows.append({
        "model": "Model 2 - Random Forest",
        "MAE": model_2_test["MAE"],
        "RMSE": model_2_test["RMSE"],
        "R2": model_2_test["R2"],
    })

model_3_test = model_3_metrics[model_3_metrics["dataset"] == "test"].iloc[0]
comparison_rows.append({
    "model": "Model 3 - HistGradientBoosting",
    "MAE": model_3_test["MAE"],
    "RMSE": model_3_test["RMSE"],
    "R2": model_3_test["R2"],
})

comparison = pd.DataFrame(comparison_rows)
display(comparison)


,model,MAE,RMSE,R2
0,Naive 168-hour historical baseline,9.307390,24.056583,0.877566
1,Model 1 - Linear Regression,7.486082,15.703616,0.947828
2,Model 2 - Random Forest,6.799299,15.259347,0.950739
3,Model 3 - HistGradientBoosting,5.784115,13.163182,0.963343


## 10. Optional permutation importance on a test sample

HistGradientBoosting does not expose the same `feature_importances_` attribute as Random Forest. This cell therefore uses permutation importance on a manageable sample of the test set. Run it after the main model result is confirmed; it is an interpretation aid, not a causal analysis.


In [22]:
from sklearn.inspection import permutation_importance

IMPORTANCE_SAMPLE_SIZE = min(20000, len(X_test))
importance_sample = X_test.sample(
    n=IMPORTANCE_SAMPLE_SIZE,
    random_state=RANDOM_STATE
)
importance_target = y_test.loc[importance_sample.index]

perm = permutation_importance(
    model_3,
    importance_sample,
    importance_target,
    n_repeats=2,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

permutation_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(permutation_df)


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.che

,feature,importance_mean,importance_std
9,trip_count_lag_1h,20.091259,0.025834
10,trip_count_lag_24h,4.270597,0.050343
13,trip_count_roll_24h,3.752417,0.076242
11,trip_count_lag_168h,3.509731,0.007710
0,PULocationID,2.294395,0.052522
1,hour,2.047833,0.044719
12,trip_count_roll_3h,0.888094,0.023115
2,day_of_week,0.389264,0.018999
4,apparent_temperature,0.039798,0.006668
8,is_holiday,0.034585,0.005397


## 11. Save Model 3 outputs


In [24]:
prediction_output = test_df[["PULocationID", "pickup_hour", "trip_count"]].copy()
prediction_output["prediction"] = test_pred
prediction_output["error"] = prediction_output["trip_count"] - prediction_output["prediction"]
prediction_output["absolute_error"] = prediction_output["error"].abs()

metrics_output = model_3_metrics.copy()
metrics_output["model"] = "Model 3 - HistGradientBoosting"
metrics_output = metrics_output[["model", "dataset", "MAE", "RMSE", "R2", "rows"]]

MODEL_PATH = MODELING_DIR / "model_3_hist_gradient_boosting.joblib"
PREDICTIONS_PATH = MODELING_DIR / "model_3_predictions.csv"
METRICS_PATH = MODELING_DIR / "model_3_metrics.csv"
IMPORTANCE_PATH = MODELING_DIR / "model_3_permutation_importance.csv"

joblib.dump(model_3, MODEL_PATH)
prediction_output.to_csv(PREDICTIONS_PATH, index=False)
metrics_output.to_csv(METRICS_PATH, index=False)
permutation_df.to_csv(IMPORTANCE_PATH, index=False)

print("Saved:")
print(" -", MODEL_PATH)
print(" -", PREDICTIONS_PATH)
print(" -", METRICS_PATH)
print(" -", IMPORTANCE_PATH)


Saved:
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_3_hist_gradient_boosting.joblib
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_3_predictions.csv
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_3_metrics.csv
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/models/modeling/model_3_permutation_importance.csv


# Model 3 complete

Record the **test-set MAE, RMSE and R²**. At this point you will have the complete first-pass comparison:

- Naive 168-hour baseline
- Model 1 — Linear Regression
- Model 2 — Random Forest
- Model 3 — HistGradientBoosting

Do **not** tune yet. First compare the three models on the same test period. Then decide whether tuning is justified.
